In [2]:
from transformers import pipeline
from tqdm import tqdm
import pandas as pd

In [6]:
sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment'
)

# Function to extract sentiment
def get_sentiment(text, max_len=512):
    text = text[:max_len]  # Truncate if too long
    result = sentiment_pipeline(text)[0]
    label = result['label']        # e.g., '2 stars'
    score = result['score']        # confidence
    stars = int(label[0])
    
    if stars <= 2: sentiment = 'Negative'
    elif stars == 3: sentiment = 'Neutral'
    else: sentiment = 'Positive'
    
    return sentiment, stars, score


NameError: name 'torch' is not defined

In [ ]:
df = pd.read_csv("../data/processed/preprocessed_reviews.csv")


# Apply to dataframe
tqdm.pandas()
df[['sentiment', 'predicted_stars', 'confidence']] = df['text_for_sentiment'].progress_apply(
    lambda x: pd.Series(get_sentiment(x))
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Map actual star ratings to sentiment
def rating_to_sentiment(r):
    if r <= 2: return 'Negative'
    elif r == 3: return 'Neutral'
    else: return 'Positive'

df['actual_sentiment'] = df['star_rating'].apply(rating_to_sentiment)

# Classification metrics
print(classification_report(df['actual_sentiment'], df['sentiment']))

In [ ]:
import matplotlib.pyplot as plt

sentiment_by_cat = df.groupby(['product_category', 'sentiment']).size().unstack(fill_value=0)

# Stacked bar chart
sentiment_by_cat.plot(kind='bar', stacked=True, colormap='RdYlGn')
plt.title('Sentiment Distribution by Product Category')
plt.tight_layout()
plt.savefig('sentiment_by_category.png')